## **Add super vision to your agent**

In [ ]:
from langchain.tools import tool, ToolRuntime

In [4]:
## prepare mocking tools
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email form the given address."""

    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subjects and body."""

    # fake email sending.
    return f"Email sent"

In [13]:
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval."
        )
    ]
)

In [22]:
config = {
    "configurable": {
        "thread_id": "chat_3"
    }
}


response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read the email and send the response.")],
        "email": "Hi Alison, i'm avilable at 12 pm can you meet that time. regards, Al Amin"
    },
    config
)


In [23]:
from pprint import pprint
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'Al '
                                                                          'Amin,\n'
                                                                          '\n'
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'your '
                                                                          'email. '
                                                                          'Yes, '
                                                                          'I '
                                                                          'can '
                        

In [16]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi Al Amin,\n\nThank you for your email. I confirm that I will meet you at 12 PM.\n\nLooking forward to it.\n\nBest regards,\nAlison'}, 'description': "Tool execution requires approval.\n\nTool: send_email\nArgs: {'body': 'Hi Al Amin,\\n\\nThank you for your email. I confirm that I will meet you at 12 PM.\\n\\nLooking forward to it.\\n\\nBest regards,\\nAlison'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='67c08d1948dda8017d1d92283cf21de7')]


## **Approve the tool call**

In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': "Hi Alison, i'm avilable at 12 pm can you meet that time. regards, "
          'Al Amin',
 'messages': [HumanMessage(content='Please read the email and send the response.', additional_kwargs={}, response_metadata={}, id='f718ceee-2bd3-4523-9b30-ab95a982b067'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_inK560yeULpj8WqsRuvuOdqx', 'function': {'arguments': '{}', 'name': 'read_email'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 76, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-Ct6mPWjzhVo2NirxJckmW5RdoJz6Y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0

## **Rejects the permission**

In [17]:
response = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "messages": "No i can meet 11 am because i've another schedule at 12 pm"
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'Al '
                                                                          'Amin,\n'
                                                                          '\n'
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'your '
                                                                          'email. '
                                                                          'I '
                                                                          'confirm '
                                                                          'that '
                    

In [20]:
print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])

Hi Al Amin,

Thank you for your email. I confirm that I will meet you at 12 PM.

Looking forward to it.

Best regards,
Alison


## **Edit**

In [24]:
response = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        # tool name to call.
                        # will usually be the same as the original actions.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This meeting is over. you're fired!!"},
                    }
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'email': "Hi Alison, i'm avilable at 12 pm can you meet that time. regards, "
          'Al Amin',
 'messages': [HumanMessage(content='Please read the email and send the response.', additional_kwargs={}, response_metadata={}, id='2f7145da-9fc0-4b5e-a130-f30fcee38671'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_wghzR4kzEjPYWZKCaUmxbLI9', 'function': {'arguments': '{}', 'name': 'read_email'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 76, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-Ct8KgfEuQUcRXEEezrzxuceaOthcm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0